# Ответы на экзаменационные вопросы: Pandas

Ноутбук закрывает вопросы **7, 8, 9, 10**, а также практические задачи из билетов
**№19 (задача 2)**, **№14 (задача 2)** и **№9 (задача 4)**.

Используемые датасеты (сгенерированы синтетически в `exam/make_datasets.py`, не являются копией
реальных наборов данных):

- `datasets/titanic.csv`
- `datasets/sp500hst.txt`
- `datasets/себестоимость_в1.xlsx`


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import make_datasets

DATA = make_datasets.DATA_DIR
make_datasets.make_all()

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


Готово. Файлы в /Users/msikanov/PycharmProjects/ege-informatics/exam/datasets:
 - addres-book-q.xml
 - anna_synthetic.txt
 - countries-of-the-world.csv
 - litw-win.csv
 - sp500hst.txt
 - titanic.csv
 - себестоимость_в1.xlsx


## Вопрос 7. Организация Pandas DataFrame и индексация для DataFrame и Series

### Организация

- **`Series`** — одномерный массив (обёртка над `numpy.ndarray`/расширенными типами) с
  **явным** индексом (`Index`), который может быть не только целочисленным по умолчанию, но и
  произвольным (строки, даты, ...). Пара "значение + метка индекса" всегда идут вместе.
- **`DataFrame`** — двумерная таблица: набор `Series`, объединённых общим `Index` по строкам
  (`.index`) и общим `Index` по столбцам (`.columns`). Фактически — словарь именованных столбцов
  одной длины, каждый из которых может иметь свой `dtype`.
- Внутри столбцы одного типа физически могут храниться в едином блоке памяти (block manager) —
  отсюда, например, эффективность работы с числовыми таблицами.

### Индексация

- `df['col']` / `df[['col1','col2']]` — доступ к столбцам по имени (или списку имён);
- `df.loc[...]` — индексация **по меткам** (значениям индекса/имени столбца), включает правую
  границу срезов;
- `df.iloc[...]` — индексация **по позиции** (целочисленной, как в NumPy), правая граница
  среза не включается;
- булево маскирование `df[df['col'] > 0]` — как и в NumPy, отбор строк по условию;
- `df.at[label, col]` / `df.iat[i, j]` — быстрый доступ к одиночному значению;
- составной (`MultiIndex`) индекс позволяет индексировать по нескольким уровням меток одновременно
  (иерархические данные).

In [2]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="values")
print(s, "\n")

df = pd.DataFrame({
    "city": ["Москва", "Казань", "Омск", "Тверь"],
    "population": [13_000_000, 1_300_000, 1_150_000, 400_000],
    "region": ["ЦФО", "ПФО", "СФО", "ЦФО"],
}, index=["r1", "r2", "r3", "r4"])
print(df)

print("\ndf['population'] (доступ к столбцу):\n", df["population"])
print("\ndf.loc['r2', 'city'] (по меткам):", df.loc["r2", "city"])
print("df.iloc[1, 0] (по позиции):        ", df.iloc[1, 0])
print("\ndf.loc[df['population'] > 1_000_000] (булева маска):\n", df.loc[df["population"] > 1_000_000])


a    10
b    20
c    30
Name: values, dtype: int64 

      city  population region
r1  Москва    13000000    ЦФО
r2  Казань     1300000    ПФО
r3    Омск     1150000    СФО
r4   Тверь      400000    ЦФО

df['population'] (доступ к столбцу):
 r1    13000000
r2     1300000
r3     1150000
r4      400000
Name: population, dtype: int64

df.loc['r2', 'city'] (по меткам): Казань
df.iloc[1, 0] (по позиции):         Казань

df.loc[df['population'] > 1_000_000] (булева маска):
       city  population region
r1  Москва    13000000    ЦФО
r2  Казань     1300000    ПФО
r3    Омск     1150000    СФО


## Вопрос 8. Универсальные функции и работа с пустыми значениями в Pandas

### Универсальные функции

Ufunc из NumPy (`np.sqrt`, `np.exp`, арифметика `+ - * /`) применяются к `Series`/`DataFrame`
поэлементно так же, как к массивам NumPy, но с важным отличием: при операции между двумя объектами
Pandas **сначала выравнивает их по индексу** (index alignment) — сопоставляет элементы не по
позиции, а по значению метки, и подставляет `NaN` там, где метка есть только в одном из операндов.

### Пустые значения (`NaN`/`None`/`NaT`)

Pandas использует `NaN` (для чисел), `None`/`NaT` (для объектов/дат) как единое представление
отсутствующих данных. Основные операции:

- `isna()` / `notna()` — маска пропусков;
- `dropna()` — удалить строки/столбцы с пропусками;
- `fillna(value)` — заполнить пропуски константой, методом (`ffill`/`bfill`) или результатом
  агрегации (например, `df['col'].fillna(df['col'].mean())`);
- большинство агрегирующих методов (`sum`, `mean`, ...) по умолчанию **пропускают** `NaN`
  (`skipna=True`), в отличие от "сырых" операций NumPy, где `NaN` заражает результат.

In [3]:
a = pd.Series([1, 2, np.nan, 4], index=["a", "b", "c", "d"])
b = pd.Series([10, 20, 30], index=["b", "c", "e"])

print("a + b (выравнивание по индексу, несовпадающие метки -> NaN):\n", a + b)

print("\na.isna():\n", a.isna())
print("\na.fillna(a.mean()) (пропуск заменён средним по остальным значениям):\n", a.fillna(a.mean()))
print("\na.sum() игнорирует NaN по умолчанию:", a.sum(), " | np.sum(a.to_numpy()) даёт NaN:", np.sum(a.to_numpy()))


a + b (выравнивание по индексу, несовпадающие метки -> NaN):
 a     NaN
b    12.0
c     NaN
d     NaN
e     NaN
dtype: float64

a.isna():
 a    False
b    False
c     True
d    False
dtype: bool

a.fillna(a.mean()) (пропуск заменён средним по остальным значениям):
 a    1.000000
b    2.000000
c    2.333333
d    4.000000
dtype: float64

a.sum() игнорирует NaN по умолчанию: 7.0  | np.sum(a.to_numpy()) даёт NaN: nan


## Вопрос 9. Объединение данных из нескольких DataFrame

### `pd.concat` — конкатенация

Склеивает объекты вдоль оси (`axis=0` — по строкам "друг под другом", `axis=1` — по столбцам
"рядом"), без анализа значений ключевых столбцов — только по совпадению индекса (при
`axis=1`) или "как есть" (при `axis=0`, с `join='outer'`/`'inner'` по столбцам).

### `pd.merge` / `DataFrame.join` — соединение по ключу (аналог SQL JOIN)

Сопоставляет строки по значению одного или нескольких столбцов-ключей (`on=`). Типы соединения
(`how=`):

- `inner` — только строки, у которых ключ есть в обеих таблицах;
- `left` / `right` — все строки одной из таблиц + совпавшие из другой (несовпавшие поля — `NaN`);
- `outer` — объединение ключей обеих таблиц (несовпавшие поля — `NaN`).

`DataFrame.join` — то же самое, но по умолчанию соединяет по **индексу**, а не по столбцу.

In [4]:
orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "customer_id": [1, 2, 2, 3], "amount": [100, 200, 150, 300]})
customers = pd.DataFrame({"customer_id": [1, 2, 4], "name": ["Иванов", "Петров", "Сидоров"]})

print("concat по строкам (axis=0):\n", pd.concat([orders.iloc[:2], orders.iloc[2:]], axis=0))

print("\nmerge how='inner' (только совпавшие customer_id):\n", pd.merge(orders, customers, on="customer_id", how="inner"))
print("\nmerge how='left' (все заказы, даже если клиент неизвестен):\n", pd.merge(orders, customers, on="customer_id", how="left"))
print("\nmerge how='outer' (объединение по всем customer_id из обеих таблиц):\n", pd.merge(orders, customers, on="customer_id", how="outer"))


concat по строкам (axis=0):
    order_id  customer_id  amount
0         1            1     100
1         2            2     200
2         3            2     150
3         4            3     300

merge how='inner' (только совпавшие customer_id):
    order_id  customer_id  amount    name
0         1            1     100  Иванов
1         2            2     200  Петров
2         3            2     150  Петров

merge how='left' (все заказы, даже если клиент неизвестен):
    order_id  customer_id  amount    name
0         1            1     100  Иванов
1         2            2     200  Петров
2         3            2     150  Петров
3         4            3     300     NaN

merge how='outer' (объединение по всем customer_id из обеих таблиц):
    order_id  customer_id  amount     name
0       1.0            1   100.0   Иванов
1       2.0            2   200.0   Петров
2       3.0            2   150.0   Петров
3       4.0            3   300.0      NaN
4       NaN            4     NaN  Сидоров


## Вопрос 10. GroupBy и подход «разбиение, применение, объединение» (split-apply-combine)

`df.groupby(key)` реализует классическую трёхшаговую схему:

1. **Split** — строки таблицы разбиваются на группы по значению одного/нескольких столбцов-ключей
   (физически строится словарь: значение ключа → список индексов строк группы).
2. **Apply** — к каждой группе независимо применяется функция: агрегирование (`mean`, `sum`, `.agg(...)`),
   преобразование (`transform`, результат той же длины, что и группа) или фильтрация (`filter`).
3. **Combine** — результаты по всем группам собираются обратно в единый объект (`Series`/`DataFrame`),
   индексированный значениями ключа группировки.

Важно: сам вызов `groupby()` **не выполняет вычислений** — он лениво возвращает объект
`DataFrameGroupBy`, вычисления происходят только при вызове конкретного метода (`.mean()`, `.agg()`, ...).

In [5]:
sales = pd.DataFrame({
    "city": ["Москва", "Москва", "Казань", "Казань", "Омск"],
    "category": ["еда", "техника", "еда", "еда", "техника"],
    "amount": [100, 500, 80, 120, 300],
})
print(sales)

g = sales.groupby("city")
print("\ntype(groupby(...)):", type(g), "— вычислений ещё нет")
print("\nsplit-apply-combine: средний чек по городу\n", g["amount"].mean())

print("\nгруппировка по двум ключам + несколько агрегаций сразу:\n",
      sales.groupby(["city", "category"])["amount"].agg(["sum", "mean", "count"]))

print("\ntransform (результат той же длины, что исходная таблица) — доля продажи в общей сумме по городу:")
sales["share_in_city"] = sales["amount"] / sales.groupby("city")["amount"].transform("sum")
print(sales)


     city category  amount
0  Москва      еда     100
1  Москва  техника     500
2  Казань      еда      80
3  Казань      еда     120
4    Омск  техника     300

type(groupby(...)): <class 'pandas.api.typing.DataFrameGroupBy'> — вычислений ещё нет

split-apply-combine: средний чек по городу
 city
Казань    100.0
Москва    300.0
Омск      300.0
Name: amount, dtype: float64

группировка по двум ключам + несколько агрегаций сразу:
                  sum   mean  count
city   category                   
Казань еда       200  100.0      2
Москва еда       100  100.0      1
       техника   500  500.0      1
Омск   техника   300  300.0      1

transform (результат той же длины, что исходная таблица) — доля продажи в общей сумме по городу:
     city category  amount  share_in_city
0  Москва      еда     100       0.166667
1  Москва  техника     500       0.833333
2  Казань      еда      80       0.400000
3  Казань      еда     120       0.600000
4    Омск  техника     300       1.000000


## Практика: билет №19, задача 2

> Датасет: `titanic.csv`. Заменить все пропущенные числовые значения возраста на значение, равное
> среднему возрасту для представителей того же класса пассажиров и того же пола (**не** выполнять
> замену, если неизвестен и класс, и пол пассажира тоже — по условию билета операция не
> выполняется, если неизвестен возраст **и** (класс **или** пол); реализуем именно так: замена
> происходит только когда класс и пол известны). Решить без циклов Python.

In [6]:
titanic = pd.read_csv(DATA / "titanic.csv")
print("пропуски по столбцам:\n", titanic.isna().sum())
print("\nвсего строк:", len(titanic))

# среднее по группам (Pclass, Sex) - без циклов, средствами groupby + transform
group_mean_age = titanic.groupby(["Pclass", "Sex"])["Age"].transform("mean")

can_fill = titanic["Age"].isna() & titanic["Pclass"].notna() & titanic["Sex"].notna()
print("\nстрок с пропущенным возрастом:", titanic['Age'].isna().sum())
print("из них поддаются восстановлению (известны класс И пол):", can_fill.sum())

titanic_filled = titanic.copy()
titanic_filled.loc[can_fill, "Age"] = group_mean_age[can_fill]

print("\nпропусков в возрасте после заполнения:", titanic_filled["Age"].isna().sum(),
      "(остались только те, где неизвестен класс или пол)")
titanic_filled.head(10)


пропуски по столбцам:
 PassengerId     0
Survived        0
Pclass          6
Name            0
Sex             6
Age            97
SibSp           0
Parch           0
Fare            0
dtype: int64

всего строк: 400

строк с пропущенным возрастом: 97
из них поддаются восстановлению (известны класс И пол): 94

пропусков в возрасте после заполнения: 3 (остались только те, где неизвестен класс или пол)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare
0,1,0,3.0,"Brown, James",male,27.900000,0,1,4.00
1,2,0,3.0,"Johnson, Николай",male,26.267021,3,2,8.87
2,3,0,3.0,"Brown, Дмитрий",male,34.500000,0,0,18.65
3,4,0,1.0,"Williams, Иван",male,16.600000,0,0,109.44
4,5,1,2.0,"Johnson, Сергей",NaN,38.700000,0,0,22.26
5,6,0,3.0,"Davis, George",male,24.500000,0,0,16.07
6,7,1,1.0,"Davis, Наталья",female,37.800000,0,0,44.78
7,8,1,1.0,"Miller, William",male,38.200000,0,0,92.58
8,9,1,2.0,"Иванов, William",male,43.000000,1,0,22.88
9,10,1,3.0,"Brown, Ольга",female,25.403226,1,2,13.26


## Практика: билет №14, задача 2

> Датасет: `sp500hst.txt` (без заголовка: дата, тикер, open, high, low, close, volume). Для
> одинаковых значений тикера рассчитать средние `open/high/low/close` за 2010 год и сохранить в
> новый CSV файл со столбцами `Тикер, open, high, low, closing`. Без циклов Python.

In [7]:
sp500 = pd.read_csv(
    DATA / "sp500hst.txt",
    header=None,
    names=["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"],
)
sp500["Date"] = pd.to_datetime(sp500["Date"], format="%Y%m%d")
print(sp500.head())

yearly_avg = (
    sp500.groupby("Ticker")[["Open", "High", "Low", "Close"]]
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={"Ticker": "Тикер", "Open": "open", "High": "high", "Low": "low", "Close": "closing"})
)
print("\nсредние значения по тикерам за 2010 год:\n", yearly_avg)

out_path = DATA / "sp500hst_yearly_avg.csv"
yearly_avg.to_csv(out_path, index=False)
print(f"\nсохранено в {out_path}")

# самопроверка чтения
pd.read_csv(out_path)


        Date Ticker   Open   High    Low  Close    Volume
0 2010-01-04   AAPL  28.83  28.95  28.66  28.84  11559128
1 2010-01-05   AAPL  28.18  28.22  28.16  28.22   6706417
2 2010-01-06   AAPL  28.11  28.27  27.89  28.19  15720178
3 2010-01-07   AAPL  28.11  28.15  28.04  28.06   9380278
4 2010-01-08   AAPL  27.85  27.89  27.78  27.83  15730664

средние значения по тикерам за 2010 год:
   Тикер    open    high     low  closing
0  AAPL   26.41   26.54   26.30    26.41
1    GE   15.14   15.21   15.07    15.14
2   IBM  126.57  127.14  126.00   126.58
3   JPM   31.91   32.04   31.76    31.91
4    KO   63.53   63.80   63.23    63.52
5  MSFT   23.75   23.85   23.65    23.74
6   PFE   20.41   20.50   20.32    20.41
7   XOM   60.09   60.35   59.84    60.08

сохранено в /Users/msikanov/PycharmProjects/ege-informatics/exam/datasets/sp500hst_yearly_avg.csv


,Тикер,open,high,low,closing
0,AAPL,26.41,26.54,26.30,26.41
1,GE,15.14,15.21,15.07,15.14
2,IBM,126.57,127.14,126.00,126.58
3,JPM,31.91,32.04,31.76,31.91
4,KO,63.53,63.80,63.23,63.52
5,MSFT,23.75,23.85,23.65,23.74
6,PFE,20.41,20.50,20.32,20.41
7,XOM,60.09,60.35,59.84,60.08


## Практика: билет №9, задача 4

> Датасет: `себестоимость_в1.xlsx`. В верхней таблице для строки «Средний физический расход
> ресурсов» пересчитать значения так, чтобы они стали равны среднему значению по рецептурам,
> представленным в таблице.

In [8]:
sebestoimost = pd.read_excel(DATA / "себестоимость_в1.xlsx")
print("исходная таблица:\n", sebestoimost)

target_row = "Средний физический расход ресурсов"
mask_resources = sebestoimost["Ресурс"] != target_row      # строки-ресурсы (без итоговой строки)
recipe_cols = sebestoimost.columns[1:]                        # все столбцы-рецептуры

recalculated = sebestoimost.copy()
recalculated.loc[sebestoimost["Ресурс"] == target_row, recipe_cols] = (
    sebestoimost.loc[mask_resources, recipe_cols].mean(axis=0).values
)

print("\nпересчитанная таблица (строка со средним физическим расходом обновлена):\n", recalculated)

out_path = DATA / "себестоимость_в1_recalculated.xlsx"
recalculated.to_excel(out_path, index=False)
print(f"\nсохранено в {out_path}")

# самопроверка: значение в итоговой строке действительно равно среднему по столбцу
check = np.allclose(
    recalculated.loc[recalculated["Ресурс"] == target_row, recipe_cols].values,
    sebestoimost.loc[mask_resources, recipe_cols].mean(axis=0).values,
)
print("проверка пройдена:", check)


исходная таблица:
                                Ресурс  Рецептура 1  Рецептура 2  Рецептура 3  Рецептура 4
0                                Мука         4.95         2.64         4.28         1.16
1                               Сахар         1.57         4.96         0.30         0.73
2                                Яйца         1.41         3.82         3.56         0.97
3                               Масло         4.79         2.26         3.29         3.99
4                              Молоко         4.09         0.46         1.71         4.62
5                                Соль         4.55         3.29         2.00         2.34
6  Средний физический расход ресурсов         0.00         0.00         0.00         0.00

пересчитанная таблица (строка со средним физическим расходом обновлена):
                                Ресурс  Рецептура 1  Рецептура 2  Рецептура 3  Рецептура 4
0                                Мука         4.95        2.640     4.280000     1.160000
1     